# A0 W&B Run Plots

A0-only notebook backed by `wandb_metrics.py`. It loads and plots the three A0 config folders: `a0_hvg`, `a0_sparse16`, and `a0_aib`.


In [25]:
from pathlib import Path
import importlib
import sys

for candidate in [Path.cwd(), *Path.cwd().parents]:
    helpers = candidate / "helpers"
    repo_helpers = candidate / "Topology_Task" / "analysis" / "metrics" / "helpers"
    if helpers.exists() and (helpers / "wandb_metrics.py").exists():
        sys.path.insert(0, str(helpers))
        break
    if repo_helpers.exists() and (repo_helpers / "wandb_metrics.py").exists():
        sys.path.insert(0, str(repo_helpers))
        break
else:
    raise FileNotFoundError("Could not locate Topology_Task/analysis/metrics/helpers")

import wandb_metrics as wm
wm = importlib.reload(wm)
print("wandb_metrics:", wm.__file__)


wandb_metrics: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/analysis/metrics/helpers/wandb_metrics.py


## Load Selected W&B Histories


In [26]:
data = wm.load_wandb_data()

runs_df = data.runs_df
history_df = data.history_df
#runs_df


Project: corentin-plumet-epfl/Grid2Op
Task dir: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task
Cache mode: full
Cache dir: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/wandb_cache
Local-only mode: True
Force refresh: False
Refresh scan-history fallbacks: False
Selected 306 cached runs from /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/wandb_cache/full_history
state
finished    202
crashed      99
killed        5
History artifact setup: local_only=True, runs_df=306
[ 1/306] loading artifact cache: a0_aib_00_flat_local_t020_s0
    loaded 359 rows, 148 columns from /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/run_data/a0_aib_cpu/runs/a0_aib_00_flat_local_t020_s0__MAPPO_bus14_T_0_0__I__1782784223_20854/history.parquet in 0.1s
[ 2/306] loading artifact cache: a0_aib_00_flat_local_t020_s0
    loaded 356 rows, 148 columns from /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/run_data/a0_aib_gpu/runs

## A0 Heuristic vs Gate


Assuming baseline = `a0_hvg_00_baseline`.

| Run family | Displayed name | Eval heuristic | Intervention gate | Gate eval mode | Rho threshold | Difference vs `a0_hvg_00_baseline` |
|---|---|---|---|---|---:|---|
| `a0_hvg_00_baseline` | baseline | `none` | `false` | `-` | `0.90` | Baseline: plain A0 policy, no heuristic override and no learned gate |
| `a0_hvg_01_eval_rho090` | global rho heuristic | `rho_threshold` | `false` | `-` | `0.90` | Adds global rho-threshold heuristic at evaluation time |
| `a0_hvg_04_eval_local_rho090` | local rho heuristic | `local_rho_threshold` | `false` | `-` | `0.90` | Adds local/per-line rho-threshold heuristic at evaluation time |
| `a0_hvg_02_gate_final_map` | gate final-action MAP | `none` | `true` | `final_action_map` | `0.90` | Adds learned intervention gate with final-action MAP evaluation |
| `a0_hvg_03_gate_hierarchical` | gate hierarchical greedy | `none` | `true` | `hierarchical_greedy` | `0.90` | Adds learned intervention gate with hierarchical greedy evaluation |

Shared unless noted: MLP actor/critic, `[256,256,256]` / `[256,256,256]`, `norm_reward = true`, `init_do_nothing_prob = 0.0`, `entropy_coef = 0.01`, `total_timesteps = 20000000`, `eval_freq = 80000`.


In [27]:
a0_hvg_result = wm.plot_a0_hvg_survival()
a0_hvg_result["fig"]


Plot source folders used to build curves: 1 folder(s), 15 run(s)
  - /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/run_data/a0_hvg (15 runs)
Saved plot: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/wandb_figures/a0_hvg_survival_baseline_comparisons.html


## A0 HVG Heuristic Curves Together

Baseline, global rho heuristic, and local rho heuristic on one graph. Each curve is the mean over seeds `s0`, `s1`, `s2`; faint lines show the individual seeds.


In [28]:
import pandas as pd

A0_HVG_SEEDS = (0, 1, 2)


def seeded_runs(prefix, seeds=A0_HVG_SEEDS):
    return [f"{prefix}_s{seed}" for seed in seeds]


def report_missing_runs(groups, history):
    available = set(wm.available_run_names(history=history))
    rows = []
    for label, runs in groups.items():
        missing = [run for run in runs if run not in available]
        rows.append({"curve": label, "expected": len(runs), "available": len(runs) - len(missing), "missing": missing})
    coverage = pd.DataFrame(rows)
    display(coverage)
    return coverage


def style_bottom_right_legend(fig, save_name=None):
    fig.update_layout(
        legend=dict(
            x=0.985,
            y=0.035,
            xanchor="right",
            yanchor="bottom",
            orientation="v",
            font=dict(size=18),
            bgcolor="rgba(255,255,255,0.88)",
            bordercolor="rgba(80,80,80,0.30)",
            borderwidth=1,
            itemsizing="constant",
            tracegroupgap=8,
            title=dict(font=dict(size=18)),
        ),
    )
    if save_name:
        wm.save_plot(fig, save_name)
    return fig


a0_hvg_heuristic_runs = {
    "baseline": seeded_runs("a0_hvg_00_baseline"),
    "global rho heuristic": seeded_runs("a0_hvg_01_eval_rho090"),
    "local rho heuristic": seeded_runs("a0_hvg_04_eval_local_rho090"),
}
a0_hvg_heuristic_coverage = report_missing_runs(a0_hvg_heuristic_runs, history_df)

fig_a0_hvg_heuristic_curves = wm.plot_run_mean_groups(
    {
        "A0 HVG: baseline vs heuristic overrides": {
            "baseline": wm.mean_curve(
                a0_hvg_heuristic_runs["baseline"],
                color="#1f77b4",
                width=4,
                member_alpha=0.16,
                std_alpha=0.10,
            ),
            "global rho heuristic": wm.mean_curve(
                a0_hvg_heuristic_runs["global rho heuristic"],
                color="#ff7f0e",
                width=4,
                member_alpha=0.16,
                std_alpha=0.12,
            ),
            "local rho heuristic": wm.mean_curve(
                a0_hvg_heuristic_runs["local rho heuristic"],
                color="#2ca02c",
                width=4,
                member_alpha=0.16,
                std_alpha=0.12,
            ),
        }
    },
    split="test",
    smooth=5,
    title="A0 HVG: baseline vs heuristic overrides",
    ncols=1,
    subplot_height=560,
    width=1350,
    y_range=[0, 105],
    show_members=True,
    show_std=True,
    save_name="a0_hvg_heuristic_curves_together",
    history=history_df,
)
fig_a0_hvg_heuristic_curves = style_bottom_right_legend(
    fig_a0_hvg_heuristic_curves,
    save_name="a0_hvg_heuristic_curves_together",
)
fig_a0_hvg_heuristic_curves


,curve,expected,available,missing
0,baseline,3,3,[]
1,global rho heuristic,3,3,[]
2,local rho heuristic,3,3,[]


Plot source folders used to build curves: 1 folder(s), 9 run(s)
  - /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/run_data/a0_hvg (9 runs)
Saved plot: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/wandb_figures/a0_hvg_heuristic_curves_together.html
Saved plot: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/wandb_figures/a0_hvg_heuristic_curves_together.html


## A0 HVG Gate Curves Together

Baseline, gate final-action MAP, and gate hierarchical greedy on one graph. Each curve is the mean over seeds `s0`, `s1`, `s2`; faint lines show the individual seeds.


In [29]:
a0_hvg_gate_runs = {
    "baseline": seeded_runs("a0_hvg_00_baseline"),
    "gate final-action MAP": seeded_runs("a0_hvg_02_gate_final_map"),
    "gate hierarchical greedy": seeded_runs("a0_hvg_03_gate_hierarchical"),
}
a0_hvg_gate_coverage = report_missing_runs(a0_hvg_gate_runs, history_df)

fig_a0_hvg_gate_curves = wm.plot_run_mean_groups(
    {
        "A0 HVG: baseline vs learned gate variants": {
            "baseline": wm.mean_curve(
                a0_hvg_gate_runs["baseline"],
                color="#1f77b4",
                width=4,
                member_alpha=0.16,
                std_alpha=0.10,
            ),
            "gate final-action MAP": wm.mean_curve(
                a0_hvg_gate_runs["gate final-action MAP"],
                color="#9467bd",
                width=4,
                member_alpha=0.16,
                std_alpha=0.12,
            ),
            "gate hierarchical greedy": wm.mean_curve(
                a0_hvg_gate_runs["gate hierarchical greedy"],
                color="#d62728",
                width=4,
                member_alpha=0.16,
                std_alpha=0.12,
            ),
        }
    },
    split="test",
    smooth=5,
    title="A0 HVG: baseline vs learned gate variants",
    ncols=1,
    subplot_height=560,
    width=1350,
    y_range=[0, 105],
    show_members=True,
    show_std=True,
    save_name="a0_hvg_gate_curves_together",
    history=history_df,
)
fig_a0_hvg_gate_curves = style_bottom_right_legend(
    fig_a0_hvg_gate_curves,
    save_name="a0_hvg_gate_curves_together",
)
fig_a0_hvg_gate_curves


,curve,expected,available,missing
0,baseline,3,3,[]
1,gate final-action MAP,3,3,[]
2,gate hierarchical greedy,3,3,[]


Plot source folders used to build curves: 1 folder(s), 9 run(s)
  - /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/run_data/a0_hvg (9 runs)
Saved plot: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/wandb_figures/a0_hvg_gate_curves_together.html
Saved plot: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/wandb_figures/a0_hvg_gate_curves_together.html


## Intervention penalty (A0 Sparse16) (changing topology has a cost) 

Assuming baseline = `a0_sparse16_flat_p000`.

| Run family | Displayed name | Intervention gate | Intervention penalty | Gate eval mode | Difference vs `a0_sparse16_flat_p000` |
|---|---|---|---:|---|---|
| `a0_sparse16_flat_p000` | flat p0.000 | `false` | `0.000` | `-` | Baseline: flat action setup with no intervention penalty and no gate |
| `a0_sparse16_flat_p001` | flat p0.001 | `false` | `0.001` | `-` | Flat with small action/intervention penalty |
| `a0_sparse16_flat_p003` | flat p0.003 | `false` | `0.003` | `-` | Flat with medium action/intervention penalty |
| `a0_sparse16_flat_p010` | flat p0.010 | `false` | `0.010` | `-` | Flat with large action/intervention penalty |
| `a0_sparse16_gated_p000` | gated p0.000 | `true` | `0.000` | `final_action_map` | Adds intervention gate without intervention penalty |
| `a0_sparse16_gated_p001` | gated p0.001 | `true` | `0.001` | `final_action_map` | Adds intervention gate plus small intervention penalty |
| `a0_sparse16_gated_p003` | gated p0.003 | `true` | `0.003` | `final_action_map` | Adds intervention gate plus medium intervention penalty |
| `a0_sparse16_gated_p010` | gated p0.010 | `true` | `0.010` | `final_action_map` | Adds intervention gate plus large intervention penalty |

Shared unless noted: MLP actor/critic, `[256,256,256]` / `[256,256,256]`, `norm_reward = true`, `init_do_nothing_prob = 0.0`, `entropy_coef = 0.01`, `total_timesteps = 20000000`, `eval_freq = 80000`.


In [30]:
a0_sparse16_result = wm.plot_a0_sparse16_survival("all")
a0_sparse16_result["fig"]

Plot source folders used to build curves: 2 folder(s), 48 run(s)
  - /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/run_data/a0_sparse16_cpu (24 runs)
  - /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/run_data/a0_sparse16_gpu (24 runs)
Saved plot: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/wandb_figures/a0_sparse16_all_survival_baseline_comparisons.html


## A0 Sparse16 All Configs Together

All Sparse16 intervention-penalty configurations on one plot, using the same mean-curve style as the HVG comparison plots. Solid lines are flat policies; dashed lines are gated policies. Colors encode the intervention penalty.


In [31]:
# Use one source slice to avoid merging CPU/GPU runs with identical run names.
# Set to "cpu" if you want the CPU reruns instead.
SPARSE16_TOGETHER_SOURCE = "gpu"

SPARSE16_PENALTY_COLORS = {
    "p0.000": "#1f77b4",
    "p0.001": "#ff7f0e",
    "p0.003": "#2ca02c",
    "p0.010": "#d62728",
}


def _ensure_bottom_right_legend_helper():
    if "style_bottom_right_legend" in globals():
        return globals()["style_bottom_right_legend"]

    def _style_bottom_right_legend(fig, save_name=None):
        fig.update_layout(
            legend=dict(
                x=0.985,
                y=0.035,
                xanchor="right",
                yanchor="bottom",
                orientation="v",
                font=dict(size=18),
                bgcolor="rgba(255,255,255,0.88)",
                bordercolor="rgba(80,80,80,0.30)",
                borderwidth=1,
                itemsizing="constant",
                tracegroupgap=8,
                title=dict(font=dict(size=18)),
            ),
        )
        if save_name:
            wm.save_plot(fig, save_name)
        return fig

    return _style_bottom_right_legend


def _sparse16_seeded(prefix, seeds=(0, 1, 2)):
    return [f"{prefix}_s{seed}" for seed in seeds]


if SPARSE16_TOGETHER_SOURCE not in {"cpu", "gpu"}:
    raise ValueError('SPARSE16_TOGETHER_SOURCE must be "cpu" or "gpu" for this combined plot.')

a0_sparse16_together_history, a0_sparse16_source_groups, _ = wm._a0_filter_history_for_source(
    history_df,
    "a0_sparse16",
    SPARSE16_TOGETHER_SOURCE,
    label=f"A0 Sparse16 all-configs {SPARSE16_TOGETHER_SOURCE}",
)

a0_sparse16_together_runs = {
    "flat p0.000": _sparse16_seeded("a0_sparse16_flat_p000"),
    "flat p0.001": _sparse16_seeded("a0_sparse16_flat_p001"),
    "flat p0.003": _sparse16_seeded("a0_sparse16_flat_p003"),
    "flat p0.010": _sparse16_seeded("a0_sparse16_flat_p010"),
    "gated p0.000": _sparse16_seeded("a0_sparse16_gated_p000"),
    "gated p0.001": _sparse16_seeded("a0_sparse16_gated_p001"),
    "gated p0.003": _sparse16_seeded("a0_sparse16_gated_p003"),
    "gated p0.010": _sparse16_seeded("a0_sparse16_gated_p010"),
}

a0_sparse16_together_coverage = report_missing_runs(
    a0_sparse16_together_runs,
    a0_sparse16_together_history,
)

def _sparse16_penalty_curve(label, *, dash="solid"):
    penalty = label.rsplit(" ", 1)[-1]
    return wm.mean_curve(
        a0_sparse16_together_runs[label],
        color=SPARSE16_PENALTY_COLORS[penalty],
        dash=dash,
        width=4,
        member_alpha=0.14,
        std_alpha=0.10,
    )


a0_sparse16_flat_mean_curves = {
    label: _sparse16_penalty_curve(label)
    for label in ["flat p0.000", "flat p0.001", "flat p0.003", "flat p0.010"]
}
a0_sparse16_gated_mean_curves = {
    label: _sparse16_penalty_curve(label)
    for label in ["gated p0.000", "gated p0.001", "gated p0.003", "gated p0.010"]
}

legend_styler = _ensure_bottom_right_legend_helper()

a0_sparse16_flat_save_name = f"a0_sparse16_{SPARSE16_TOGETHER_SOURCE}_flat_configs_together"
fig_a0_sparse16_flat_configs_together = wm.plot_run_mean_groups(
    {
        f"A0 Sparse16 flat intervention penalty ({SPARSE16_TOGETHER_SOURCE.upper()})": a0_sparse16_flat_mean_curves,
    },
    split="test",
    smooth=5,
    title=f"A0 Sparse16 flat policies: intervention-penalty configs ({SPARSE16_TOGETHER_SOURCE.upper()})",
    ncols=1,
    subplot_height=600,
    width=1450,
    y_range=[0, 105],
    show_members=True,
    show_std=True,
    save_name=a0_sparse16_flat_save_name,
    history=a0_sparse16_together_history,
)
fig_a0_sparse16_flat_configs_together = legend_styler(
    fig_a0_sparse16_flat_configs_together,
    save_name=a0_sparse16_flat_save_name,
)


a0_sparse16_gated_save_name = f"a0_sparse16_{SPARSE16_TOGETHER_SOURCE}_gated_configs_together"
fig_a0_sparse16_gated_configs_together = wm.plot_run_mean_groups(
    {
        f"A0 Sparse16 gated intervention penalty ({SPARSE16_TOGETHER_SOURCE.upper()})": a0_sparse16_gated_mean_curves,
    },
    split="test",
    smooth=5,
    title=f"A0 Sparse16 gated policies: intervention-penalty configs ({SPARSE16_TOGETHER_SOURCE.upper()})",
    ncols=1,
    subplot_height=600,
    width=1450,
    y_range=[0, 105],
    show_members=True,
    show_std=True,
    save_name=a0_sparse16_gated_save_name,
    history=a0_sparse16_together_history,
)
fig_a0_sparse16_gated_configs_together = legend_styler(
    fig_a0_sparse16_gated_configs_together,
    save_name=a0_sparse16_gated_save_name,
)

display(fig_a0_sparse16_flat_configs_together)
fig_a0_sparse16_gated_configs_together


,curve,expected,available,missing
0,flat p0.000,3,3,[]
1,flat p0.001,3,3,[]
2,flat p0.003,3,3,[]
3,flat p0.010,3,3,[]
4,gated p0.000,3,3,[]
5,gated p0.001,3,3,[]
6,gated p0.003,3,3,[]
7,gated p0.010,3,3,[]


Plot source folders used to build curves: 1 folder(s), 12 run(s)
  - /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/run_data/a0_sparse16_gpu (12 runs)
Saved plot: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/wandb_figures/a0_sparse16_gpu_flat_configs_together.html
Saved plot: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/wandb_figures/a0_sparse16_gpu_flat_configs_together.html
Plot source folders used to build curves: 1 folder(s), 12 run(s)
  - /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/run_data/a0_sparse16_gpu (12 runs)
Saved plot: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/wandb_figures/a0_sparse16_gpu_gated_configs_together.html
Saved plot: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/wandb_figures/a0_sparse16_gpu_gated_configs_together.html


In [32]:
a0_sparse16_result = wm.plot_a0_sparse16_survival("cpu")
a0_sparse16_result["fig"]

Plot source folders used to build curves: 1 folder(s), 24 run(s)
  - /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/run_data/a0_sparse16_cpu (24 runs)
Saved plot: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/wandb_figures/a0_sparse16_cpu_survival_baseline_comparisons.html


In [33]:
a0_sparse16_result = wm.plot_a0_sparse16_survival(source="gpu")
a0_sparse16_result["fig"]


Plot source folders used to build curves: 1 folder(s), 24 run(s)
  - /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/run_data/a0_sparse16_gpu (24 runs)
Saved plot: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/wandb_figures/a0_sparse16_gpu_survival_baseline_comparisons.html


## A0 Adaptive Intervention Budget


Assuming plot baseline = `a0_hvg_00_baseline` (plain A0 baseline).

| Run family | Displayed name | Adaptive budget | Cost mode | Target | Rho threshold | Intervention gate | Difference vs `a0_hvg_00_baseline` |
|---|---|---|---|---:|---:|---|---|
| `a0_hvg_00_baseline` | plain A0 baseline | `false` | `-` | `-` | `-` | `false` | Plain A0 baseline: no adaptive intervention budget and no gate |
| `a0_aib_00_flat_local_t020` | flat local target 0.20 | `true` | `local_safe` | `0.20` | `0.90` | `false` | Adds adaptive intervention budget with local-safe cost and target `0.20` |
| `a0_aib_01_flat_local_t010` | flat local target 0.10 | `true` | `local_safe` | `0.10` | `0.90` | `false` | Same local-safe budget mechanism with lower target `0.10` |
| `a0_aib_02_flat_local_t035` | flat local target 0.35 | `true` | `local_safe` | `0.35` | `0.90` | `false` | Same local-safe budget mechanism with higher target `0.35` |
| `a0_aib_03_gate_hgreedy_sep_local_t020` | gate h-greedy separate entropy target 0.20 | `true` | `local_safe` | `0.20` | `0.90` | `hierarchical_greedy` | Adds intervention gate with hierarchical greedy eval and separate gate entropy while keeping local-safe target `0.20` |
| `a0_aib_04_flat_nonidle_t020` | flat non-idle target 0.20 | `true` | `nonidle` | `0.20` | `0.90` | `false` | Uses non-idle cost instead of local-safe cost at target `0.20` |

Shared unless noted: MLP actor/critic, `[256,256,256]` / `[256,256,256]`, `norm_reward = true`, `init_do_nothing_prob = 0.0`, `entropy_coef = 0.01`, `total_timesteps = 20000000`, `eval_freq = 80000`.


## A0 AIB Curves Together

Adaptive-intervention-budget configurations in the same mean-curve style as the HVG and Sparse16 comparison plots. Each curve is the mean over seeds `s0`, `s1`, `s2`; faint lines show individual seeds. The first plot compares the flat budget variants against the plain A0 baseline; the second compares the gated AIB variant against the matched flat local target `0.20` and the same baseline.


In [38]:
# Use one source slice to avoid merging CPU/GPU runs with identical run names.
# Set to "cpu" if you want the CPU reruns instead.
AIB_TOGETHER_SOURCE = "cpu"

AIB_VARIANT_COLORS = {
    "plain A0 baseline": "#1f77b4",
    "flat local target 0.20": "#ff7f0e",
    "flat local target 0.10": "#2ca02c",
    "flat local target 0.35": "#d62728",
    "flat non-idle target 0.20": "#9467bd",
    "gate h-greedy target 0.20": "#8c564b",
}


def _aib_seeded(prefix, seeds=(0, 1, 2)):
    return [f"{prefix}_s{seed}" for seed in seeds]


if AIB_TOGETHER_SOURCE not in {"cpu", "gpu"}:
    raise ValueError('AIB_TOGETHER_SOURCE must be "cpu" or "gpu" for this combined plot.')

a0_aib_budget_history, a0_aib_source_groups, _ = wm._a0_filter_history_for_source(
    history_df,
    "a0_aib",
    AIB_TOGETHER_SOURCE,
    label=f"A0 AIB all-configs {AIB_TOGETHER_SOURCE}",
)
a0_aib_baseline_history, a0_aib_baseline_groups, _ = wm._a0_filter_history_for_source(
    history_df,
    "a0_hvg",
    AIB_TOGETHER_SOURCE,
    allow_unsplit_fallback=True,
    label=f"A0 AIB baseline {AIB_TOGETHER_SOURCE}",
)
a0_aib_together_history = pd.concat(
    [a0_aib_budget_history, a0_aib_baseline_history],
    ignore_index=True,
)
if {"run_id", "_step"}.issubset(a0_aib_together_history.columns):
    a0_aib_together_history = a0_aib_together_history.drop_duplicates(["run_id", "_step"])

a0_aib_together_runs = {
    "plain A0 baseline": _aib_seeded("a0_hvg_00_baseline"),
    "flat local target 0.20": _aib_seeded("a0_aib_00_flat_local_t020"),
    "flat local target 0.10": _aib_seeded("a0_aib_01_flat_local_t010"),
    "flat local target 0.35": _aib_seeded("a0_aib_02_flat_local_t035"),
    "flat non-idle target 0.20": _aib_seeded("a0_aib_04_flat_nonidle_t020"),
    "gate h-greedy target 0.20": _aib_seeded("a0_aib_03_gate_hgreedy_sep_local_t020"),
}

a0_aib_together_coverage = report_missing_runs(
    a0_aib_together_runs,
    a0_aib_together_history,
)


def _aib_mean_curve(label, *, dash="solid"):
    return wm.mean_curve(
        a0_aib_together_runs[label],
        history=a0_aib_together_history,
        color=AIB_VARIANT_COLORS[label],
        dash=dash,
        width=4,
        member_alpha=0.14,
        std_alpha=0.10,
    )


legend_styler = _ensure_bottom_right_legend_helper()

a0_aib_flat_labels = [
    "plain A0 baseline",
    "flat local target 0.20",
    "flat local target 0.10",
    "flat local target 0.35",
    "flat non-idle target 0.20",
]
a0_aib_flat_save_name = f"a0_aib_{AIB_TOGETHER_SOURCE}_flat_budget_configs_together"
fig_a0_aib_flat_configs_together = wm.plot_run_mean_groups(
    {
        f"A0 AIB flat budget variants ({AIB_TOGETHER_SOURCE.upper()})": {
            label: _aib_mean_curve(label)
            for label in a0_aib_flat_labels
        },
    },
    split="test",
    smooth=5,
    title=f"A0 AIB flat budget variants vs plain baseline ({AIB_TOGETHER_SOURCE.upper()})",
    ncols=1,
    subplot_height=600,
    width=1450,
    y_range=[0, 105],
    show_members=True,
    show_std=True,
    save_name=a0_aib_flat_save_name,
    history=a0_aib_together_history,
)
fig_a0_aib_flat_configs_together = legend_styler(
    fig_a0_aib_flat_configs_together,
    save_name=a0_aib_flat_save_name,
)

a0_aib_gate_labels = [
    "plain A0 baseline",
    "flat local target 0.20",
    "gate h-greedy target 0.20",
]
a0_aib_gate_save_name = f"a0_aib_{AIB_TOGETHER_SOURCE}_gate_budget_configs_together"
fig_a0_aib_gate_configs_together = wm.plot_run_mean_groups(
    {
        f"A0 AIB gated budget variant ({AIB_TOGETHER_SOURCE.upper()})": {
            "plain A0 baseline": _aib_mean_curve("plain A0 baseline"),
            "flat local target 0.20": _aib_mean_curve("flat local target 0.20"),
            "gate h-greedy target 0.20": _aib_mean_curve("gate h-greedy target 0.20", dash="dash"),
        },
    },
    split="test",
    smooth=5,
    title=f"A0 AIB gate h-greedy vs matched flat budget ({AIB_TOGETHER_SOURCE.upper()})",
    ncols=1,
    subplot_height=600,
    width=1450,
    y_range=[0, 105],
    show_members=True,
    show_std=True,
    save_name=a0_aib_gate_save_name,
    history=a0_aib_together_history,
)
fig_a0_aib_gate_configs_together = legend_styler(
    fig_a0_aib_gate_configs_together,
    save_name=a0_aib_gate_save_name,
)

display(fig_a0_aib_flat_configs_together)
fig_a0_aib_gate_configs_together


,curve,expected,available,missing
0,plain A0 baseline,3,3,[]
1,flat local target 0.20,3,3,[]
2,flat local target 0.10,3,3,[]
3,flat local target 0.35,3,3,[]
4,flat non-idle target 0.20,3,3,[]
5,gate h-greedy target 0.20,3,3,[]


Plot source folders used to build curves: 2 folder(s), 15 run(s)
  - /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/run_data/a0_aib_cpu (12 runs)
  - /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/run_data/a0_hvg (3 runs)
Saved plot: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/wandb_figures/a0_aib_cpu_flat_budget_configs_together.html
Saved plot: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/wandb_figures/a0_aib_cpu_flat_budget_configs_together.html
Plot source folders used to build curves: 2 folder(s), 9 run(s)
  - /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/run_data/a0_aib_cpu (6 runs)
  - /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/run_data/a0_hvg (3 runs)
Saved plot: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/wandb_figures/a0_aib_cpu_gate_budget_configs_together.html
Saved plot: /Users/corentinplumet/Documents/RL_Marl2grid/Topolog

In [35]:
a0_aib_result = wm.plot_a0_aib_survival("all")
a0_aib_result["fig"]

Plot source folders used to build curves: 3 folder(s), 33 run(s)
  - /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/run_data/a0_aib_cpu (15 runs)
  - /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/run_data/a0_aib_gpu (15 runs)
  - /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/run_data/a0_hvg (3 runs)
Saved plot: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/wandb_figures/a0_aib_all_survival_vs_plain_a0_baseline.html


In [36]:
a0_aib_result = wm.plot_a0_aib_survival("cpu")
a0_aib_result["fig"]

Plot source folders used to build curves: 2 folder(s), 18 run(s)
  - /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/run_data/a0_aib_cpu (15 runs)
  - /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/run_data/a0_hvg (3 runs)
Saved plot: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/wandb_figures/a0_aib_cpu_survival_vs_plain_a0_baseline.html


In [37]:
a0_aib_result = wm.plot_a0_aib_survival("gpu")
a0_aib_result["fig"]

Plot source folders used to build curves: 2 folder(s), 18 run(s)
  - /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/run_data/a0_aib_gpu (15 runs)
  - /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/run_data/a0_hvg (3 runs)
Saved plot: /Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task/outputs/wandb_figures/a0_aib_gpu_survival_vs_plain_a0_baseline.html
